# Segmentation Model with Simulated Data

## **Load Data**

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

db = pd.read_csv('nanopore_trace_database_v2.csv')
train_db = db[db['trace_id'] < 500]
test_db  = db[db['trace_id'] >= 500]


In [3]:
db.info()

<class 'pandas.DataFrame'>
RangeIndex: 2778233 entries, 0 to 2778232
Data columns (total 5 columns):
 #   Column            Dtype  
---  ------            -----  
 0   time_ms           float64
 1   current_pA        float64
 2   region            str    
 3   trace_id          int64  
 4   peptide_sequence  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 106.0 MB


## **Define the Transition Matrix**

In [4]:
import numpy as np

transmat = np.array([
    [0.9997, 0.0003, 0.0000],  # from DNA
    [0.0000, 0.8000, 0.2000],  # from linker
    [0.0000, 0.0000, 1.0000],  # from peptide
])

startprob = np.array([1, 0, 0])  # starting probabilities

## **Define Emmission Distributions**

In [5]:
# For DNA

dna_samples = train_db[train_db['region'] == 'DNA']
dna_mean = dna_samples['current_pA'].mean()
dna_std = dna_samples['current_pA'].std()


# For linker

linker_samples = train_db[train_db['region'] == 'linker']
linker_mean = linker_samples['current_pA'].mean()
linker_std = linker_samples['current_pA'].std()

# for peptide
peptide_samples = train_db[train_db['region'] == 'peptide']
peptide_mean = peptide_samples['current_pA'].mean()
peptide_std = peptide_samples['current_pA'].std()


means = [dna_mean, linker_mean, peptide_mean]
stds = [dna_std, linker_std, peptide_std]







print(f"DNA: Mean = {dna_mean:.2f} pA, Std Dev = {dna_std:.2f} pA")
print(f"Linker: Mean = {linker_mean:.2f} pA, Std Dev = {linker_std:.2f} pA")
print(f"Peptide: Mean = {peptide_mean:.2f} pA, Std Dev = {peptide_std:.2f} pA")

DNA: Mean = 78.69 pA, Std Dev = 17.06 pA
Linker: Mean = 98.54 pA, Std Dev = 18.63 pA
Peptide: Mean = 112.23 pA, Std Dev = 9.01 pA


### Visualize how the distributions look, evaluate the difficulty of this problem

In [6]:
for region in ['DNA', 'linker', 'peptide']:
    subset = db[db['region'] == region]['current_pA']
    plt.hist(subset, bins=50, alpha=0.5, label=region)
    plt.xlabel('Current (pA)')
    plt.ylabel('Frequency')
    plt.title('Current Distribution by Region')
plt.legend()
plt.show()

ModuleNotFoundError: No module named 'matplotlib_inline'

## **Run HMM: store parameters and compute emission probabilities**

In [ ]:
from hmmlearn.hmm import GaussianHMM

REGION_TO_STATE = {'DNA': 0, 'linker': 1, 'peptide': 2}
STATE_TO_REGION = {0: 'DNA', 1: 'linker', 2: 'peptide'}

means_arr  = np.array(means).reshape(3, 1)
covars_arr = np.array(stds).reshape(3, 1, 1) ** 2

model = GaussianHMM(n_components=3, covariance_type='full', n_iter=0)
model.n_features  = 1               # must be set before covars_ in this hmmlearn version
model.startprob_  = startprob.astype(float)
model.transmat_   = transmat
model.means_      = means_arr
model.covars_     = covars_arr

print("means_   (pA) :", model.means_.flatten())
print("covars_  (pA²):", model.covars_.flatten())
print("startprob     :", model.startprob_)
print("transmat      :\n", model.transmat_)

## **Run Viterbi to Find the Max Probability**

In [ ]:
accuracies  = {}
predictions = {}

for tid in test_db['trace_id'].unique():
    trace = test_db[test_db['trace_id'] == tid]
    X = trace['current_pA'].values.reshape(-1, 1)

    pred_states = model.predict(X)
    true_states = trace['region'].map(REGION_TO_STATE).values

    acc = np.mean(pred_states == true_states)
    accuracies[tid]  = acc
    predictions[tid] = {
        'time_ms':     trace['time_ms'].values,
        'current_pA':  trace['current_pA'].values,
        'true_states': true_states,
        'pred_states': pred_states,
    }

overall_acc = np.mean(list(accuracies.values()))
print(f"Overall accuracy: {overall_acc:.2%}\n")
print(f"{'Trace':>6}  {'Accuracy':>8}")
for tid, acc in sorted(accuracies.items()):
    print(f"  {tid:>6}  {acc:>8.2%}")

## **Visualize**

In [ ]:
import matplotlib.patches as mpatches

REGION_COLORS = {'DNA': '#4895ef', 'linker': '#48cae4', 'peptide': '#f72585'}

def shade_regions(ax, t, states, ylo, yhi, alpha=0.85):
    for state, region in STATE_TO_REGION.items():
        mask = states == state
        if mask.any():
            ax.fill_between(t, ylo, yhi, where=mask,
                            color=REGION_COLORS[region], alpha=alpha, linewidth=0)

n     = len(predictions)
ncols = 5
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 2.5), squeeze=False)

for idx, (tid, data) in enumerate(predictions.items()):
    ax = axes[idx // ncols][idx % ncols]
    t, I = data['time_ms'], data['current_pA']

    ymin, ymax = I.min(), I.max()
    band = (ymax - ymin) * 0.08

    shade_regions(ax, t, data['true_states'], ymax,        ymax + band)  # top strip = true
    shade_regions(ax, t, data['pred_states'], ymin - band, ymin)          # bottom strip = predicted

    ax.plot(t, I, color='#222', lw=0.4, alpha=0.7)
    ax.set_ylim(ymin - band * 1.2, ymax + band * 1.2)
    ax.set_title(f'Trace {tid}  |  {accuracies[tid]:.1%}', fontsize=7)
    ax.tick_params(labelsize=6)
    ax.set_xlabel('ms', fontsize=6)

for idx in range(n, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

legend_patches  = [mpatches.Patch(color=c, label=r) for r, c in REGION_COLORS.items()]
legend_patches += [
    mpatches.Patch(color='gray', alpha=0.7, label='top strip = true label'),
    mpatches.Patch(color='gray', alpha=0.3, label='bottom strip = predicted'),
]
fig.legend(handles=legend_patches, loc='upper right', fontsize=8)
fig.suptitle(f'HMM Segmentation — {n} held-out traces  |  Overall accuracy: {overall_acc:.2%}',
             fontsize=11, y=1.005)
plt.tight_layout()
plt.show()

In [ ]:
import ipywidgets as widgets
from IPython.display import display

trace_ids = list(predictions.keys())

def plot_trace(idx):
    tid = trace_ids[idx]
    data = predictions[tid]
    t, I = data['time_ms'], data['current_pA']

    fig, ax = plt.subplots(figsize=(12, 3))
    ymin, ymax = I.min(), I.max()
    band = (ymax - ymin) * 0.08

    shade_regions(ax, t, data['true_states'], ymax, ymax + band)
    shade_regions(ax, t, data['pred_states'], ymin - band, ymin)

    ax.plot(t, I, color='#222', lw=0.4, alpha=0.7)
    ax.set_ylim(ymin - band * 1.2, ymax + band * 1.2)
    ax.set_title(f'Trace {tid}  |  Accuracy: {accuracies[tid]:.1%}', fontsize=10)
    ax.set_xlabel('ms')

    legend_patches = [mpatches.Patch(color=c, label=r) for r, c in REGION_COLORS.items()]
    legend_patches += [
        mpatches.Patch(color='gray', alpha=0.7, label='top = true'),
        mpatches.Patch(color='gray', alpha=0.3, label='bottom = predicted'),
    ]
    ax.legend(handles=legend_patches, fontsize=8, loc='upper right')
    plt.tight_layout()
    plt.show()

widgets.interact(plot_trace, idx=widgets.IntSlider(min=0, max=len(trace_ids)-1, step=1, description='Trace'));


In [ ]:
PEPTIDE_STATE = REGION_TO_STATE['peptide']

boundary_errors = {}

for tid, data in predictions.items():
    true_idxs = np.where(data['true_states'] == PEPTIDE_STATE)[0]
    pred_idxs = np.where(data['pred_states'] == PEPTIDE_STATE)[0]

    if len(true_idxs) == 0 or len(pred_idxs) == 0:
        continue  # peptide not found in true or predicted

    true_start_ms = data['time_ms'][true_idxs[0]]
    pred_start_ms = data['time_ms'][pred_idxs[0]]
    boundary_errors[tid] = pred_start_ms - true_start_ms  # positive = predicted late, negative = predicted early

errors = np.array(list(boundary_errors.values()))
print(f"Mean boundary error : {errors.mean():.3f} ms")
print(f"Std  boundary error : {errors.std():.3f} ms")
print(f"MAE                 : {np.abs(errors).mean():.3f} ms")
print(f"Max early           : {errors.min():.3f} ms")
print(f"Max late            : {errors.max():.3f} ms")

plt.figure(figsize=(8, 3))
plt.hist(errors, bins=30, edgecolor='black')
plt.axvline(0, color='red', linestyle='--', label='perfect boundary')
plt.xlabel('Boundary error (ms)  [+ = predicted late, − = predicted early]')
plt.ylabel('Count')
plt.title('Peptide Start Boundary Detection Error')
plt.legend()
plt.tight_layout()
plt.show()
